In [0]:
for stream in spark.streams.active:
    stream.stop()

In [0]:
stream_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .option("numRows", 50)
    .load()
)

query = (
    stream_df.writeStream
    .format("memory")
    .queryName("rate_stream_demo")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/workspace/ecommerce_lakehouse/streaming_checkpoints/rate_stream_demo_2"
    )
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
display(spark.sql("SELECT * FROM rate_stream_demo LIMIT 20"))

timestamp,value
2026-06-18T13:03:18.975Z,0
2026-06-18T13:03:19.175Z,1
2026-06-18T13:03:19.375Z,2
2026-06-18T13:03:19.575Z,3


In [0]:
from pyspark.sql.functions import *
orders_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .option("numRows", 100)
    .load()
    .withColumn("order_id", concat(lit("ORD-"), col("value")))
    .withColumn("customer_id", concat(lit("CUST-"), col("value") % 50))
    .withColumn(
        "product",
        expr("""
        CASE
            WHEN value % 6 = 0 THEN 'Laptop'
            WHEN value % 6 = 1 THEN 'Phone'
            WHEN value % 6 = 2 THEN 'Headphones'
            WHEN value % 6 = 3 THEN 'Keyboard'
            WHEN value % 6 = 4 THEN 'Mouse'
            ELSE 'Monitor'
        END
        """)
    )
    .withColumn("quantity", (col("value") % 5 + 1))
    .withColumn("amount", round(rand() * 50000 + 100, 2))
    .withColumn(
        "payment_status",
        expr("""
        CASE
            WHEN value % 3 = 0 THEN 'paid'
            WHEN value % 3 = 1 THEN 'failed'
            ELSE 'pending'
        END
        """)
    )
    .withColumnRenamed("timestamp", "event_time")
    .drop("value")
)

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS ecommerce_lakehouse.bronze_orders (
    event_time TIMESTAMP,
    order_id STRING,
    customer_id STRING,
    product STRING,
    quantity BIGINT,
    amount DOUBLE,
    payment_status STRING
)
USING DELTA
""")


DataFrame[]

In [0]:
bronze_query = (
    orders_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/workspace/ecommerce_lakehouse/streaming_checkpoints/bronze_orders"
    )
    .trigger(availableNow=True)
    .toTable("ecommerce_lakehouse.bronze_orders")
)

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM ecommerce_lakehouse.bronze_orders
        LIMIT 20
    """)
)

event_time,order_id,customer_id,product,quantity,amount,payment_status
2026-06-18T13:06:15.074Z,ORD-2,CUST-2,Headphones,3,22363.57,pending
2026-06-18T13:06:14.874Z,ORD-1,CUST-1,Phone,2,1137.8,failed
2026-06-18T13:06:14.674Z,ORD-0,CUST-0,Laptop,1,18054.08,paid


In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS ecommerce_lakehouse.silver_orders (
    event_time TIMESTAMP,
    order_id STRING,
    customer_id STRING,
    product STRING,
    quantity BIGINT,
    amount DOUBLE,
    payment_status STRING,
    processed_at TIMESTAMP
)
USING DELTA
""")

DataFrame[]

In [0]:
silver_orders_df = (
    spark.table("ecommerce_lakehouse.bronze_orders")
    .dropDuplicates(["order_id"])
    .filter(col("order_id").isNotNull())
    .filter(col("customer_id").isNotNull())
    .filter(col("product").isNotNull())
    .filter(col("quantity") > 0)
    .filter(col("amount") > 0)
    .filter(col("payment_status").isin("paid", "failed", "pending"))
    .withColumn("processed_at", current_timestamp())
)

In [0]:
(
    silver_orders_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ecommerce_lakehouse.silver_orders")
)

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM ecommerce_lakehouse.silver_orders
        LIMIT 20
    """)
)

event_time,order_id,customer_id,product,quantity,amount,payment_status,processed_at
2026-06-18T13:06:15.074Z,ORD-2,CUST-2,Headphones,3,22363.57,pending,2026-06-18T13:07:20.852Z
2026-06-18T13:06:14.874Z,ORD-1,CUST-1,Phone,2,1137.8,failed,2026-06-18T13:07:20.852Z
2026-06-18T13:06:14.674Z,ORD-0,CUST-0,Laptop,1,18054.08,paid,2026-06-18T13:07:20.852Z


In [0]:
gold_revenue_by_product = (
    spark.table("ecommerce_lakehouse.silver_orders")
    .filter(col("payment_status") == "paid")
    .groupBy("product")
    .agg(
        round(sum("amount"), 2).alias("total_revenue"),
        count("*").alias("total_orders")
    )
    .orderBy(col("total_revenue").desc())
)

gold_revenue_by_product.write.format("delta").mode("overwrite").saveAsTable(
    "ecommerce_lakehouse.gold_revenue_by_product"
)

In [0]:
display(spark.table("ecommerce_lakehouse.gold_revenue_by_product"))


product,total_revenue,total_orders
Laptop,18054.08,1


In [0]:
gold_payment_status_summary = (
    spark.table("ecommerce_lakehouse.silver_orders")
    .groupBy("payment_status")
    .agg(
        count("*").alias("transaction_count"),
        round(sum("amount"), 2).alias("total_amount")
    )
)

In [0]:
gold_payment_status_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_lakehouse.gold_payment_status_summary")

In [0]:
display(
    spark.table("ecommerce_lakehouse.gold_payment_status_summary")
)

payment_status,transaction_count,total_amount
pending,1,22363.57
paid,1,18054.08
failed,1,1137.8


In [0]:
gold_payment_status_summary = (
    spark.table("ecommerce_lakehouse.silver_orders")
    .groupBy("payment_status")
    .agg(
        count("*").alias("transaction_count"),
        round(sum("amount"), 2).alias("total_amount")
    )
)

gold_payment_status_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_lakehouse.gold_payment_status_summary")

display(
    spark.table("ecommerce_lakehouse.gold_payment_status_summary")
)

payment_status,transaction_count,total_amount
pending,1,22363.57
paid,1,18054.08
failed,1,1137.8


In [0]:
display(
    spark.sql("""
        SELECT product, total_revenue, total_orders
        FROM ecommerce_lakehouse.gold_revenue_by_product
        ORDER BY total_revenue DESC
    """)
)

product,total_revenue,total_orders
Laptop,18054.08,1


In [0]:
display(
    spark.sql("""
        SELECT payment_status, transaction_count, total_amount
        FROM ecommerce_lakehouse.gold_payment_status_summary
    """)
)

payment_status,transaction_count,total_amount
pending,1,22363.57
paid,1,18054.08
failed,1,1137.8


In [0]:
display(
    spark.sql("""
        SELECT customer_id, total_spent, order_count
        FROM ecommerce_lakehouse.gold_customer_spending
        ORDER BY total_spent DESC
        LIMIT 10
    """)
)

payment_status,transaction_count,total_amount
pending,1,22363.57
paid,1,18054.08
failed,1,1137.8


In [0]:
display(
    spark.sql("SHOW TABLES IN ecommerce_lakehouse")
)

database,tableName,isTemporary
ecommerce_lakehouse,bronze_orders,false
ecommerce_lakehouse,gold_payment_status_summary,false
ecommerce_lakehouse,gold_revenue_by_product,false
ecommerce_lakehouse,silver_orders,false
,rate_stream_demo,true


In [0]:
gold_customer_spending = (
    spark.table("ecommerce_lakehouse.silver_orders")
    .filter(col("payment_status") == "paid")
    .groupBy("customer_id")
    .agg(
        round(sum("amount"), 2).alias("total_spent"),
        count("*").alias("order_count")
    )
    .orderBy(col("total_spent").desc())
)

gold_customer_spending.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_lakehouse.gold_customer_spending")

In [0]:
gold_suspicious_orders = (
    spark.table("ecommerce_lakehouse.silver_orders")
    .filter(col("amount") > 40000)
)

gold_suspicious_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_lakehouse.gold_suspicious_orders")

In [0]:
display(spark.sql("SHOW TABLES IN ecommerce_lakehouse"))

database,tableName,isTemporary
ecommerce_lakehouse,bronze_orders,false
ecommerce_lakehouse,gold_customer_spending,false
ecommerce_lakehouse,gold_payment_status_summary,false
ecommerce_lakehouse,gold_revenue_by_product,false
ecommerce_lakehouse,gold_suspicious_orders,false
ecommerce_lakehouse,silver_orders,false
,rate_stream_demo,true
